## 面试问题

子循环嵌套（sub-loop）：内层 loop 的深度/预算/结果回传与隔离？

## 回答主线

嵌套子循环要控深度、隔离预算、只回传摘要、隔离失败。本 Notebook 演示两点：无深度限制的嵌套会一直下探到安全上限(失控)，有 max_depth 的在上限干净停止；子循环在隔离的子预算内运行，超支只返回部分结果、不透支父预算。

## 真实案例

写报告主任务里的检索子任务开子循环。父预算 100，分 40% 给子循环，深度上限 3。对比无深度限制与有限制，并展示子预算隔离。数据为教学循环，不代表真实系统。

In [1]:
PARENT_BUDGET = 100  # 父循环总预算。
SUB_BUDGET_RATIO = 0.4  # 分配给子循环的预算比例。
MAX_DEPTH = 3  # 子循环嵌套深度上限。

sub_budget = int(PARENT_BUDGET * SUB_BUDGET_RATIO)  # 计算子循环预算。
print("父预算:", PARENT_BUDGET)  # 展示父预算。
print("子循环预算:", sub_budget, "(父的 40%)")  # 展示隔离的子预算。
print("深度上限:", MAX_DEPTH)  # 展示深度上限。

父预算: 100
子循环预算: 40 (父的 40%)
深度上限: 3


## 基线（Baseline）

反面基线：无深度限制的嵌套。子任务每层都再开一层子循环，没有 `max_depth` 就一直下探，只能靠安全上限兜底——这就是失控的无限嵌套。

In [2]:
def run_nested(depth, max_depth, safety=50):  # 嵌套子循环 depth 记录当前层。
    if depth >= safety:  # 安全上限防止真正无限递归。
        return {"reached": depth, "stopped_by": "safety"}  # 命中安全上限。
    if max_depth is not None and depth >= max_depth:  # 到达设定的深度上限。
        return {"reached": depth, "stopped_by": "max_depth"}  # 正常在深度上限停止。
    return run_nested(depth + 1, max_depth, safety)  # 否则继续下探一层。

no_limit = run_nested(0, None)  # 无深度上限的嵌套。
print("无深度限制结果:", no_limit)  # 展示一直下探到安全上限即失控。

无深度限制结果: {'reached': 50, 'stopped_by': 'safety'}


## 失败案例与修正

无深度限制会失控。修正一是设 `max_depth`：到达即停。修正二是子预算隔离：子循环在自己的子预算内运行，超支返回部分结果、父预算只扣子循环实际花费。

In [3]:
with_limit = run_nested(0, MAX_DEPTH)  # 有深度上限的嵌套。
print("有深度限制结果:", with_limit)  # 展示在深度上限干净停止。

有深度限制结果: {'reached': 3, 'stopped_by': 'max_depth'}


In [4]:
def run_sub_with_budget(sub_budget, cost_per_step, max_steps=10):  # 子循环在隔离的子预算内运行。
    spent = 0  # 子循环已花预算。
    steps = 0  # 子循环步数。
    for _ in range(max_steps):  # 子循环推进。
        if spent + cost_per_step > sub_budget:  # 超出子预算即停。
            return {"steps": steps, "sub_spent": spent, "stopped_by": "sub_budget"}  # 子预算耗尽返回部分结果。
        spent += cost_per_step  # 累加子预算消耗。
        steps += 1  # 推进步数。
    return {"steps": steps, "sub_spent": spent, "stopped_by": "completed"}  # 正常完成。

sub_result = run_sub_with_budget(sub_budget, cost_per_step=15)  # 子循环每步消耗 15。
parent_remaining = PARENT_BUDGET - sub_result["sub_spent"]  # 父预算只扣掉子循环实际花费。
print("子循环结果:", sub_result)  # 展示子循环在子预算内停止。
print("父循环剩余预算:", parent_remaining)  # 展示子循环未透支父预算。

子循环结果: {'steps': 2, 'sub_spent': 30, 'stopped_by': 'sub_budget'}
父循环剩余预算: 70


In [5]:
print("无深度限制停于:", no_limit["stopped_by"], "深度", no_limit["reached"])  # 无限制下探到安全上限。
print("有深度限制停于:", with_limit["stopped_by"], "深度", with_limit["reached"])  # 有限制在深度上限停。
print("子循环花费", sub_result["sub_spent"], "父预算剩", parent_remaining, "未透支")  # 展示预算隔离。

无深度限制停于: safety 深度 50
有深度限制停于: max_depth 深度 3
子循环花费 30 父预算剩 70 未透支


## 结果解读

无深度限制一直下探到安全上限 50（失控）；有 `max_depth=3` 在深度 3 干净停止。子循环每步花 15，在子预算 40 内跑 2 步后停（花 30），父预算只被扣 30、仍剩 70——预算隔离成立。要点：深度硬约束、子预算不透支父、只回传摘要、失败当动作失败。

In [6]:
assert no_limit["stopped_by"] == "safety"  # 无深度限制一直下探到安全上限。
assert with_limit["stopped_by"] == "max_depth"  # 有深度限制在设定上限停止。
assert with_limit["reached"] == 3  # 深度停在上限 3。
assert sub_result["sub_spent"] <= sub_budget  # 子循环不超子预算。
assert parent_remaining == 70  # 父预算只扣子循环实际花费。
print("全部不变量通过")  # 输出测试通过信号。

全部不变量通过
